In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torchaudio
from matplotlib.patches import Rectangle

PROJECT_DIR = Path.cwd().parent
SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from core.config import P, settings
from core.setup import setup_logging, setup_project_path

setup_logging(settings.LOG_LEVEL)
setup_project_path(PROJECT_DIR)
# setup_data()

CLEANED_DIR = settings.data_dir / "cleaned"
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

In [ ]:
from core.config import Parameters

BoxSource = pd.DataFrame | Path | None

# Una grabacion a la vez: ~54 MB de espectrograma por minuto de audio con los P actuales.
_SPEC_CACHE: dict[tuple, np.ndarray] = {}


def full_spec_db(wav_path: Path, params: Parameters = P) -> np.ndarray:
    """Espectrograma en dB de la grabacion completa, remuestreada a params.target_sr.

    Se cachea solo la ultima grabacion calculada, para que reejecutar el visor sobre
    el mismo archivo sea instantaneo sin acumular RAM.
    """
    key = (str(wav_path), params.target_sr, params.n_fft, params.win_length, params.hop_length)
    if key not in _SPEC_CACHE:
        waveform, native_sr = torchaudio.load(wav_path)
        waveform = waveform.mean(dim=0)
        if native_sr != params.target_sr:
            waveform = torchaudio.functional.resample(waveform, native_sr, params.target_sr)
        spec = torchaudio.transforms.Spectrogram(
            n_fft=params.n_fft,
            win_length=params.win_length,
            hop_length=params.hop_length,
            power=2.0,
        )(waveform)
        _SPEC_CACHE.clear()
        _SPEC_CACHE[key] = (10 * torch.log10(spec + params.eps)).numpy()
    return _SPEC_CACHE[key]


def db_baseline(spec_db: np.ndarray, percentiles: tuple[float, float] = (5.0, 99.5)):
    """Rango (lo, hi) en dB de referencia, sobre una submuestra de toda la grabacion.

    Se calcula una sola vez por grabacion: asi el mapeo de color no cambia al hacer
    pan o zoom, igual que en Raven.
    """
    lo, hi = np.percentile(spec_db[::4, ::8], percentiles)
    return float(lo), float(hi)


def db_clim(
    baseline: tuple[float, float], brightness: float = 0.0, contrast: float = 1.0
) -> tuple[float, float]:
    """Convierte brillo/contraste en el rango (vmin, vmax) que se pasa a la imagen.

    - brightness (dB): desplaza el centro del rango; positivo => imagen mas clara.
    - contrast (x): comprime el rango alrededor del centro; > 1 => mas contraste.
    """
    lo, hi = baseline
    center = 0.5 * (lo + hi) - brightness
    half_range = 0.5 * (hi - lo) / max(contrast, 1e-3)
    return center - half_range, center + half_range


def _load_boxes(source: BoxSource) -> pd.DataFrame:
    if source is None:
        return pd.DataFrame()
    return source if isinstance(source, pd.DataFrame) else pd.read_csv(source, sep="\t")


def _draw_boxes(ax, rows: pd.DataFrame, color: str) -> list:
    """Dibuja las cajas en tiempo absoluto y devuelve sus artistas, para poder borrarlas."""
    artists = []
    for _, row in rows.iterrows():
        x0, y0 = row["Begin Time (s)"], row["Low Freq (Hz)"]
        width, height = row["End Time (s)"] - x0, row["High Freq (Hz)"] - y0
        artists.append(
            ax.add_patch(
                Rectangle((x0, y0), width, height, edgecolor=color, facecolor="none", linewidth=1.5)
            )
        )
        label = f"{row['Species']}/{row['Call type']}"
        if "Score" in row:
            label += f" {row['Score']:.2f}"
        artists.append(
            ax.text(x0, y0 + height, label, color=color, fontsize=9, fontweight="bold", va="bottom")
        )
    return artists

In [ ]:
%matplotlib widget

import ipywidgets as widgets
from IPython.display import display

ZOOM_STEPS_S = [3.0, 5.0, 10.0, 15.0, 20.0, 30.0, 60.0]


def explore_clips(
    wav_path: Path,
    annotations: BoxSource = None,
    detections: BoxSource = None,
    params: Parameters = P,
    cmap: str = "magma",
    max_cols: int = 2000,
) -> None:
    spec_db = full_spec_db(wav_path, params)
    baseline = db_baseline(spec_db)
    n_frames = spec_db.shape[1]
    sr, hop = params.target_sr, params.hop_length
    duration_s = (n_frames - 1) * hop / sr
    nyquist_hz = sr / 2

    box_sources = {
        "GT": (_load_boxes(annotations), "cyan"),
        "modelo": (_load_boxes(detections), "lime"),
    }
    box_sources = {name: v for name, v in box_sources.items() if not v[0].empty}
    scored = any("Score" in rows.columns for rows, _ in box_sources.values())

    layout = widgets.Layout(width="70%")
    zooms = [z for z in ZOOM_STEPS_S if z < duration_s] + [round(duration_s, 2)]
    w_zoom = widgets.SelectionSlider(
        options=[(f"{z:g} s", z) for z in zooms],
        value=zooms[0],
        description="Zoom",
        layout=layout,
    )
    w_time = widgets.FloatSlider(
        value=0.0,
        min=0.0,
        max=duration_s,
        step=0.05,
        description="Inicio (s)",
        readout_format=".2f",
        layout=layout,
    )
    w_brightness = widgets.FloatSlider(
        value=0.0, min=-60.0, max=60.0, step=1.0, description="Brillo (dB)", layout=layout
    )
    w_contrast = widgets.FloatSlider(
        value=1.0, min=0.2, max=5.0, step=0.05, description="Contraste", layout=layout
    )
    w_score = widgets.FloatSlider(
        value=0.5, min=0.0, max=1.0, step=0.01, description="Score >=", layout=layout
    )

    with plt.ioff():  # sin esto ipympl muestra la figura ademas del display() de abajo
        fig, ax = plt.subplots(figsize=(11, 6))
    fig.canvas.header_visible = False
    fig.canvas.toolbar_position = "right"
    im = ax.imshow(
        spec_db[:, :1],
        origin="lower",
        aspect="auto",
        extent=(0.0, duration_s, 0.0, nyquist_hz),
        cmap=cmap,
    )
    if box_sources:
        ax.legend(
            handles=[
                Rectangle((0, 0), 1, 1, edgecolor=color, facecolor="none", label=name)
                for name, (_, color) in box_sources.items()
            ],
            loc="lower right",
            fontsize=9,
        )
    ax.set_xlabel("Tiempo (s)")
    ax.set_ylabel("Frecuencia (Hz)")
    ax.set_title(Path(wav_path).name)
    box_artists: list = []

    def move(_=None) -> None:
        """Recorta la ventana visible del espectrograma ya calculado."""
        t0 = min(w_time.value, max(duration_s - w_zoom.value, 0.0))
        t1 = min(t0 + w_zoom.value, duration_s)
        i0, i1 = int(t0 * sr / hop), max(int(t1 * sr / hop), int(t0 * sr / hop) + 1)
        step = max(1, (i1 - i0) // max_cols)  # no dibujar mas columnas que pixeles utiles
        im.set_data(spec_db[:, i0:i1:step])
        im.set_extent((i0 * hop / sr, i1 * hop / sr, 0.0, nyquist_hz))
        ax.set_xlim(t0, t1)
        fig.canvas.draw_idle()

    def rezoom(change=None) -> None:
        """Reajusta el rango del slider de tiempo y hace zoom sobre el centro de la vista."""
        previous_zoom = change["old"] if change else w_zoom.value
        center = w_time.value + previous_zoom / 2
        w_time.max = max(duration_s - w_zoom.value, 0.0)
        w_time.step = max(w_zoom.value / 50, 0.01)
        w_time.value = min(max(center - w_zoom.value / 2, 0.0), w_time.max)
        move()

    def restretch(_=None) -> None:
        im.set_clim(*db_clim(baseline, w_brightness.value, w_contrast.value))
        fig.canvas.draw_idle()

    def refilter(_=None) -> None:
        """Redibuja las cajas que superan el umbral de score."""
        while box_artists:
            box_artists.pop().remove()
        for rows, color in box_sources.values():
            visible = rows[rows["Score"] >= w_score.value] if "Score" in rows.columns else rows
            box_artists.extend(_draw_boxes(ax, visible, color))
        fig.canvas.draw_idle()

    w_zoom.observe(rezoom, names="value")
    w_time.observe(move, names="value")
    w_brightness.observe(restretch, names="value")
    w_contrast.observe(restretch, names="value")
    w_score.observe(refilter, names="value")
    rezoom()
    restretch()
    refilter()

    controls = [w_zoom, w_time, w_brightness, w_contrast]
    if scored:
        controls.append(w_score)
    display(widgets.VBox(controls), fig.canvas)

In [ ]:
from infer import load_model
from pipelines.inference_pipeline import predict

device = "cuda" if torch.cuda.is_available() else "cpu"
checkpoint_path = PROJECT_DIR / "checkpoints" / "128d_64q_iouiou_10cls_best.pth"
model, labels = load_model(checkpoint_path, device)

# wav_path = CLEANED_DIR / "weddells_saddleBack_tamarin__LW/240125_0028.wav"
# ann_path = wav_path.with_suffix(".txt")

# detections = predict(model, wav_path, labels, device, score_threshold=0.8)
# explore_clips(wav_path, annotations=ann_path, detections=detections)

In [ ]:
wav_path = settings.data_dir / "test_audio/20240214_102444.wav"
# ann_path = wav_path.with_suffix(".txt")
detections = predict(model, wav_path, labels, device, score_threshold=0.2)
# detections = wav_path.with_suffix(".selections.txt")
explore_clips(wav_path, annotations=None, detections=detections)